## 准备输入文件

本 notebook 使用标准 Jupyter 接口，可在 Colab、AutoDL、本地或自建 GPU 服务器运行。

项目根目录 `PROJECT_ROOT` 下需要：

| 文件 | 作用 | 是否必需 |
| --- | --- | --- |
| `ramanv2/` 或 `ramanv2.zip` | 代码包 | 二选一 |
| `data.npz` | 恢复 `dataset/<数据集>/...` 下的 `init`、`init_test` 与 `dataset.json` / `.raman_pipeline/dataset_mapping.json` | 可选，目录已在位时可省略 |

`PROJECT_ROOT` 自动探测：`RAMAN_PROJECT_ROOT` 环境变量 / 显式覆盖 → 从当前工作目录向上找到含 `ramanv2/` 或 `ramanv2.zip` 的目录 → 当前工作目录。

AutoDL 建议：
- 把项目放到数据盘，例如 `/root/autodl-tmp/raman`，并在该目录启动 Jupyter；否则在下一个单元设置 `PROJECT_ROOT_OVERRIDE`。
- 首次运行会自动安装缺失的 scipy / scikit-learn / pandas / seaborn，默认使用清华镜像。若已开启 AutoDL 学术加速，可把 `PIP_INDEX_URL` 设为 `None`。
- `umap-learn` 依赖 numba/llvmlite，默认不装；需要嵌入分析时把 `INSTALL_UMAP_DEPS` 设为 `True`。
- 如果 pip 网络受限，把 `INSTALL_MISSING_DEPS` 设为 `False`，按单元打印的命令在终端手动安装后重启 kernel。
- 如果直接编辑了 `ramanv2/`，请删除 `ramanv2.zip` 或同步更新 zip，否则解包单元会用 zip 覆盖本地改动。

In [ ]:
"""运行环境引导：定位项目根目录、校验依赖、报告设备信息。"""

import importlib.util
import os
import subprocess
import sys
from pathlib import Path


# ==================== 可编辑参数 ====================
# None 表示自动探测；AutoDL 可写死为 "/root/autodl-tmp/raman"。
PROJECT_ROOT_OVERRIDE = None
# 缺失依赖时自动 pip 安装；网络受限时设为 False，按提示手动安装。
INSTALL_MISSING_DEPS = True
# AutoDL 默认清华镜像；Colab/海外可设为 None 使用默认 PyPI。
PIP_INDEX_URL = (
    os.environ.get("PIP_INDEX_URL")
    or os.environ.get("RAMAN_PIP_INDEX_URL")
    or "https://pypi.tuna.tsinghua.edu.cn/simple"
)
# umap-learn 会拉取 numba/llvmlite，首次安装可能较慢；仅嵌入分析需要。
INSTALL_UMAP_DEPS = True
PIP_TIMEOUT_SECONDS = 900

PACKAGE_ARCHIVE_NAME = "ramanv2.zip"
DATA_ARCHIVE_NAME = "data.npz"
# 训练、评估、分析链路需要的第三方模块 -> pip 包名。
REQUIRED_MODULES = {
    "torch": "torch",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "yaml": "PyYAML",
    "openpyxl": "openpyxl",
}
# 嵌入分析才需要的可选依赖，避免首次运行被 numba/llvmlite 编译阻塞。
OPTIONAL_MODULES = {"umap": "umap-learn"}


def is_module_available(name: str) -> bool:
    """安全探测模块能否导入；find_spec 对多级名称在父包缺失时会抛异常。"""
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def has_project_payload(directory: Path) -> bool:
    """目录是否已含可用代码包：ramanv2.zip，或含 core/config.py 的 ramanv2/ 包。"""
    if (directory / PACKAGE_ARCHIVE_NAME).is_file():
        return True
    return (directory / "ramanv2" / "core" / "config.py").is_file()


def resolve_project_root() -> Path:
    """按 环境变量/显式覆盖 -> 工作目录及其祖先 的顺序定位项目根目录。"""
    override = PROJECT_ROOT_OVERRIDE or os.environ.get("RAMAN_PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        root.mkdir(parents=True, exist_ok=True)
        return root
    working_dir = Path.cwd().resolve()
    for candidate in (working_dir, *working_dir.parents):
        if has_project_payload(candidate):
            return candidate
    return working_dir


def _pip_command(packages: list[str]) -> list[str]:
    """构建带镜像、超时和进度显示的 pip 命令。"""
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--progress-bar",
        "on",
        "--timeout",
        "60",
        "--retries",
        "2",
    ]
    if PIP_INDEX_URL:
        command.extend(["--index-url", PIP_INDEX_URL])
    return command + list(packages)


def _install_packages(packages: list[str]) -> bool:
    """执行 pip 安装；失败或超时时打印可复制的手动命令。"""
    command = _pip_command(packages)
    print("执行：" + " ".join(command))
    try:
        subprocess.run(
            command,
            shell=False,
            check=True,
            text=True,
            encoding="utf-8",
            timeout=PIP_TIMEOUT_SECONDS,
        )
        return True
    except subprocess.TimeoutExpired:
        print(f"pip 超过 {PIP_TIMEOUT_SECONDS} 秒未完成，已停止等待。")
    except subprocess.CalledProcessError as exc:
        print(f"pip 安装失败（exit={exc.returncode}）。")
    print("请在终端手动执行以下命令，然后重启 kernel：")
    print("  " + " ".join(command))
    return False


def ensure_dependencies(install_enable: bool) -> list[str]:
    """检查必需模块并按需 pip 安装，返回仍然缺失的核心模块名。"""
    missing = [name for name in REQUIRED_MODULES if not is_module_available(name)]
    optional_missing = [name for name in OPTIONAL_MODULES if not is_module_available(name)]
    if missing and install_enable:
        packages = [REQUIRED_MODULES[name] for name in missing]
        print(f"安装缺失依赖：{', '.join(packages)}")
        _install_packages(packages)
        missing = [name for name in REQUIRED_MODULES if not is_module_available(name)]
    elif missing:
        print(f"缺少依赖（未自动安装）：{', '.join(REQUIRED_MODULES[name] for name in missing)}")
        print("可手动执行：")
        print("  " + " ".join(_pip_command([REQUIRED_MODULES[name] for name in missing])))
    if optional_missing and install_enable and INSTALL_UMAP_DEPS:
        packages = [OPTIONAL_MODULES[name] for name in optional_missing]
        print(f"安装可选依赖：{', '.join(packages)}")
        _install_packages(packages)
        optional_missing = [name for name in OPTIONAL_MODULES if not is_module_available(name)]
    if optional_missing:
        print(
            "可选依赖未安装（仅嵌入分析需要）："
            + ", ".join(OPTIONAL_MODULES[name] for name in optional_missing)
        )
    return missing


PROJECT_ROOT = resolve_project_root()
still_missing_modules = ensure_dependencies(INSTALL_MISSING_DEPS)
print(f"project_root = {PROJECT_ROOT}")
print(f"pip_index    = {PIP_INDEX_URL or 'default PyPI'}")
if still_missing_modules:
    print(f"缺少依赖（需手动安装）：{', '.join(still_missing_modules)}")

import shutil as shutil_module

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "缺少 torch：AutoDL 镜像通常自带 CUDA 版 torch；请按实例 CUDA 版本手动安装后重启 kernel。"
    ) from exc

print(f"python       = {sys.version.split()[0]} ({sys.executable})")
print(f"torch        = {torch.__version__}, cuda_build = {torch.version.cuda}")
print(f"cpu_count    = {os.cpu_count()}")
print(
    f"disk_free    = {shutil_module.disk_usage(str(PROJECT_ROOT)).free / 2**30:.1f} GiB"
    f" @ {PROJECT_ROOT}"
)
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(
            f"gpu[{index}]      = {properties.name}, "
            f"cc={properties.major}.{properties.minor}, "
            f"vram={properties.total_memory / 2**30:.1f} GiB"
        )
    print(f"bf16_support = {torch.cuda.is_bf16_supported()}")
else:
    print("CUDA 不可用：训练与评估会回退到 CPU，use_amp 自动关闭。")
    print("AutoDL: 请确认选择的是 GPU 实例，并检查 torch 是否为 CUDA 版本。")

In [ ]:
# 只做本地检查：缺少代码包直接报错，缺少数据包由后续恢复单元决定是否必需。
REQUIRED_ARCHIVES = {"ramanv2.zip", DATA_ARCHIVE_NAME}
absent_archives = sorted(
    name for name in REQUIRED_ARCHIVES if not (PROJECT_ROOT / name).is_file()
)
if PACKAGE_ARCHIVE_NAME in absent_archives and not (PROJECT_ROOT / "ramanv2").is_dir():
    raise FileNotFoundError(
        f"缺少代码包：请把 {PACKAGE_ARCHIVE_NAME} 或解压后的 ramanv2/ 放到 {PROJECT_ROOT}"
    )
if DATA_ARCHIVE_NAME in absent_archives:
    print(f"未找到 {DATA_ARCHIVE_NAME}：若 dataset/<数据集>/init 已在位可忽略。")
print("输入检查完成。")

In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

PACKAGE_ARCHIVE = PROJECT_ROOT / PACKAGE_ARCHIVE_NAME
PACKAGE_DIR = PROJECT_ROOT / "ramanv2"


def load_package():
    """检测到压缩包时替换 ramanv2；否则使用已解压的包。"""
    if PACKAGE_ARCHIVE.is_file():
        temp_dir = Path(tempfile.mkdtemp(prefix="ramanv2_unpack_", dir=PROJECT_ROOT))
        try:
            with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
                archive.extractall(temp_dir)
            unpacked_dir = temp_dir / PACKAGE_DIR.name
            if not unpacked_dir.is_dir():
                raise FileNotFoundError("压缩文件中未找到 ramanv2/ 目录")
            if PACKAGE_DIR.exists():
                if not PACKAGE_DIR.is_dir():
                    raise FileExistsError(f"包路径不是目录：{PACKAGE_DIR}")
                shutil.rmtree(PACKAGE_DIR)
            shutil.move(str(unpacked_dir), str(PROJECT_ROOT))
            print(f"replaced package: {PACKAGE_ARCHIVE.name} -> {PACKAGE_DIR}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
    elif not PACKAGE_DIR.is_dir():
        raise FileNotFoundError(f"请先将 {PACKAGE_ARCHIVE_NAME} 放到 {PROJECT_ROOT}")
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))


load_package()


In [ ]:
import importlib
import sys


def reload_ramanv2():
    """清除 ramanv2 模块缓存，以加载当前目录中的最新代码。"""
    for name in list(sys.modules):
        if name == "ramanv2" or name.startswith("ramanv2."):
            del sys.modules[name]
    importlib.invalidate_caches()
    config_module = importlib.import_module("ramanv2.core.config")
    print("ramanv2 modules reloaded.")
    return config_module


config_module = reload_ramanv2()
build_config = config_module.build_config


In [ ]:
from ramanv2.packaging.platform_bundle import DATA_ARCHIVE_NAME, restore_data_archive

# 首次运行或更换 data.npz 后执行一次；日常改设置不需要重跑本格。
DATA_ARCHIVE = PROJECT_ROOT / DATA_ARCHIVE_NAME
if DATA_ARCHIVE.is_file():
    restore_report = restore_data_archive(DATA_ARCHIVE, PROJECT_ROOT)
    print(
        f"restored {DATA_ARCHIVE.name}: samples={restore_report['sample_count']} "
        f"stages={restore_report['sample_counts']}"
    )
else:
    print(f"未找到 {DATA_ARCHIVE.name}，使用已有 dataset 目录")


In [ ]:
from pathlib import Path

import yaml
from ramanv2.core.config import build_config
from ramanv2.data.datasets import load_dataset_registry
from ramanv2.data.profiles import resolve_selection

# 本格只决定数据集与实验级配置；可反复调整，不会重新解压数据。
# 可用值会在下方自动打印（PB=病原菌，FBB=食源菌）。
DATASET_ID = "PB"
# 填写整数时覆盖本次运行的随机种子；None 时使用 config.py 的默认值。
SEED = None
# 继续已有实验时填写实验根，例如 output/病原菌/PB/unified_20261112_hier；
# 扁平数据集例如 output/食源菌/FBB/unified_20261112。
# 实验根存在时会直接复用其中的 shared_config.yaml。
# 要新建实验，重跑本格（或留空）。
EXPERIMENT_DIR = None
LAST_TRAINED_SLOT_DIR = None
# 跨平台常用覆盖项：显存或 CPU 核数差异较大时按需打开。
# 加载进程数默认 -1，表示按 CPU 核数自动解析；显存不足时把 batch_size 降到 32。
CONFIG_OVERRIDES = {
    # "batch_size": 64,
    # "train_loader_num_workers": 8,
    # "val_loader_num_workers": 8,
}

dataset_root = PROJECT_ROOT / "dataset"
registry = load_dataset_registry(dataset_root)
for item in registry.list_datasets():
    print(
        f"{item.dataset_id:6} {item.display_name}  dir={item.directory.name}  "
        f"levels={item.hierarchy.level_names}"
    )

selection = resolve_selection(DATASET_ID, None, PROJECT_ROOT)
spec = selection.spec
profile = selection.profile
dataset_dir = selection.dataset_dir
PROFILE_ID = profile.profile_id
INIT_DIR = dataset_dir / profile.root_init
PROFILE_TEST_INIT_DIR = dataset_dir / profile.root_init_test
config_values = {
    "dataset_id": spec.dataset_id,
    "profile_id": PROFILE_ID,
    "label_schema": selection.hierarchy.label_schema,
    "bad_bands": spec.bad_bands,
}
if SEED is not None:
    config_values["seed"] = int(SEED)
config_values.update(CONFIG_OVERRIDES)
config = build_config(config_values)
if EXPERIMENT_DIR is not None:
    experiment_snapshot = Path(EXPERIMENT_DIR) / "shared_config.yaml"
    if not experiment_snapshot.is_file():
        raise FileNotFoundError(f"实验目录缺少 shared_config.yaml：{experiment_snapshot}")
    config = build_config(yaml.safe_load(experiment_snapshot.read_text(encoding="utf-8")))
    print(f"复用实验配置：{experiment_snapshot}")
print(
    f"\nselected = {spec.dataset_id} / {spec.display_name}"
    f"  hierarchy={selection.hierarchy.level_names}  bad_bands={spec.bad_bands}"
)
print(f"dataset_dir = {dataset_dir}")
print("训练目标（LEVEL_NAME / PARENT_INDEX / GLOBAL_HEAD_ENABLE）请在“训练”格设置。")

## 构建训练集

In [ ]:
from ramanv2.data.builders.train_test import build_train

# build_train 先在临时目录构建，成功后原子替换 train/；失败时保留当前产物。
build_train(profile, dataset_dir, input_config=config.input, hierarchy=selection.hierarchy)

In [ ]:
from ramanv2.data.count import count_dataset, print_count_results

tree, sample_count = count_dataset(dataset_dir / profile.root_train_clean)
print_count_results(tree, sample_count)


In [ ]:
from datetime import datetime

from ramanv2.data.experiment_group import build_experiment_group

# 为当前数据集生成共享切分的实验组；需要先完成上面的 train 构建。
# 如果不做层级就跳过，无需运行
# PB 使用 ("hier", "genus", "species")；扁平数据集例如 FBB 使用 ("main",)。
PREPARE_EXPERIMENT_GROUP_ENABLE = False # 要走层级设为 True
GROUP_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
GROUP_EXPERIMENTS = ("hier", "genus", "species")
GROUP_SEED = 42
GROUP_TRAIN_RATIO = 0.8

if PREPARE_EXPERIMENT_GROUP_ENABLE:
    group_report = build_experiment_group(
        profile_id=profile.profile_id,
        tag=GROUP_TAG,
        experiment_names=GROUP_EXPERIMENTS,
        seed=GROUP_SEED,
        train_ratio=GROUP_TRAIN_RATIO,
    )
    print(
        f"dataset = {group_report['profile_id']}  "
        f"samples = {group_report['sample_count']}  "
        f"train = {group_report['train_count']}  val = {group_report['val_count']}"
    )
    for name, item in group_report["experiments"].items():
        print(f"  {name:8} {item['experiment_dir']}")
        print(f"    train_sha256 = {item['train_split_sha256']}")
    print("训练格将 EXPERIMENT_DIR 指向对应实验目录；FBB 单实验使用 tag 目录本身。")


## 构建独立测试集

`dataset/<数据集>/init_test` 存在时从它构建 `test/`；测试集不参与训练与验证切分，用于训练完成后的独立效果判断。

In [ ]:
from ramanv2.data.builders.train_test import build_test

if PROFILE_TEST_INIT_DIR.is_dir():
    build_test(
        profile,
        dataset_dir,
        input_config=config.input,
        test_root=PROFILE_TEST_INIT_DIR,
    )
else:
    print(f"未找到 {PROFILE_TEST_INIT_DIR}，跳过独立测试集构建。")


In [ ]:
if (dataset_dir / profile.root_test).is_dir():
    test_tree, test_sample_count = count_dataset(dataset_dir / profile.root_test)
    print_count_results(test_tree, test_sample_count)
else:
    print("独立测试集目录不存在。")


## 训练

本格设置本次训练的 `LEVEL_NAME`、`PARENT_INDEX` 与 `GLOBAL_HEAD_ENABLE`：

- `GLOBAL_HEAD_ENABLE=True`：训练该层全部类别的全局模型，`level_1` 以外的层级也可用，`PARENT_INDEX` 必须为 `None`。
- 否则 `level_1` 训练全局模型；其余层级填写父类索引训练单个子模型，留空则批量训练该层全部可训练父类（单子类父类自动跳过）。

上一格会按层级列出可选父类索引、子类数量和是否需要训练。

同一实验目录只保留一套 `train_split.json` / `val_split.json`；在同一实验内切换层级或父类后重复运行本格，会复用该实验与切分。要新建实验，回到数据格重跑（`EXPERIMENT_DIR` 会重置为 `None`）或手动填写新的实验目录。

训练链路按 `torch.cuda.is_available()` 选择设备，单次运行只占用一张卡；多卡机器用 `CUDA_VISIBLE_DEVICES` 指定卡号即可。
只有一个属的数据（例如 FUNG）可以跳过 level_1，直接训练 level_2 的 `PARENT_INDEX=0`；单类顶层会直通。

In [ ]:
from ramanv2.data.runtime.index import DatasetIndex

# 训练前查看可选父类：训练 level_N 时，PARENT_INDEX 对应上一层的类别索引。
# 只有一个子类的父类不需要训练，推理时直通。
# 若训练该层全部类别的全局模型，在下一格把 GLOBAL_HEAD_ENABLE 设为 True。
train_dir = dataset_dir / profile.root_train_clean
if selection.hierarchy.is_flat or len(selection.hierarchy.level_names) < 2:
    print("当前数据集没有上级层级，PARENT_INDEX 保持 None。")
elif not train_dir.is_dir():
    print(f"训练目录不存在：{train_dir}，请先构建训练集。")
else:
    dataset_index = DatasetIndex(train_dir, selection.hierarchy, load_intensity_enable=False)
    level_names = dataset_index.level_names
    for level_index in range(1, len(level_names)):
        level_name = level_names[level_index]
        parent_level = level_names[level_index - 1]
        parent_names = dataset_index.get_class_names(parent_level)
        child_names = dataset_index.get_class_names(level_name)
        children_by_parent = dataset_index.parent_to_children.get(level_name, {})
        print(f"训练 {level_name}（父类 {parent_level}）时 PARENT_INDEX 可选值：")
        trainable_ids = []
        for parent_id, parent_name in enumerate(parent_names):
            child_ids = children_by_parent.get(parent_id, [])
            if not child_ids:
                continue
            if len(child_ids) >= 2:
                trainable_ids.append(parent_id)
            train_note = "需训练" if len(child_ids) >= 2 else "单子类直通"
            child_labels = [child_names[child_id].split("/")[-1] for child_id in child_ids]
            print(
                f"  {parent_id:>2}  {parent_name:<20}  "
                f"子类 {len(child_ids):>2}  {train_note}  {', '.join(child_labels)}"
            )
        print("可训练 PARENT_INDEX：" + ", ".join(str(item) for item in trainable_ids))
        print(f"该层全局模型类别数：{len(child_names)}")
        print()

In [ ]:
import numpy as np
from datetime import datetime

from ramanv2.common.paths import resolve_path
from ramanv2.data.runtime.index import DatasetIndex
from ramanv2.training.snapshot import TRAIN_ARCHIVE_NAME
from ramanv2.training.workflow import TrainRequest, run_training

# ===== 本次训练目标：切换层级/父类只改这里 =====
LEVEL_NAME = "level_1"
# 非首层留空表示批量训练该层全部可训练父类；填写整数表示只训练该父类；
# 单子类父类自动跳过（推理时直通）。
PARENT_INDEX = None
# True 时训练该层全部类别的全局模型，可配合 level_2/level_3 得到平铺结构；
# 此时 PARENT_INDEX 必须为 None。
GLOBAL_HEAD_ENABLE = False

if EXPERIMENT_DIR is None:
    active_experiment_dir = (
        PROJECT_ROOT
        / "output"
        / selection.spec.directory.name
        / profile.dataset_name
        / datetime.now().strftime("%Y%m%d_%H%M%S")
    )
else:
    active_experiment_dir = resolve_path(EXPERIMENT_DIR)
# 先写回，训练中途失败后重跑仍落回同一实验目录，而不是新建时间戳目录。
EXPERIMENT_DIR = active_experiment_dir


def resolve_training_parent_ids():
    """返回本次要训练的父类索引；None 表示全局模型。"""
    if GLOBAL_HEAD_ENABLE:
        if PARENT_INDEX is not None:
            raise ValueError("GLOBAL_HEAD_ENABLE=True 时必须将 PARENT_INDEX 设为 None")
        return [None]
    if LEVEL_NAME == "level_1":
        if PARENT_INDEX is not None:
            raise ValueError("level_1 必须将 PARENT_INDEX 设为 None")
        return [None]
    train_dir = dataset_dir / profile.root_train_clean
    if not train_dir.is_dir():
        raise FileNotFoundError(f"训练目录不存在：{train_dir}，请先构建训练集")
    dataset_index = DatasetIndex(train_dir, selection.hierarchy, load_intensity_enable=False)
    level_name = dataset_index.resolve_level_name(LEVEL_NAME)
    if PARENT_INDEX is not None:
        return [int(PARENT_INDEX)]
    children_by_parent = dataset_index.parent_to_children.get(level_name, {})
    return [
        parent_id
        for parent_id, child_ids in sorted(children_by_parent.items())
        if len(child_ids) >= 2
    ]


parent_ids = resolve_training_parent_ids()
split_ready = all(
    (active_experiment_dir / name).is_file()
    for name in ("train_split.json", "val_split.json")
)
print(
    f"experiment_dir = {active_experiment_dir}  "
    f"({'复用已有 split' if split_ready else '首次训练，新建 split'})"
)
if not parent_ids:
    print(f"{LEVEL_NAME} 没有需要训练的父类（单子类父类均直通）。")
    LAST_TRAINED_SLOT_DIR = None
else:
    trained_slot_dirs = []
    for parent_id in parent_ids:
        trained_slot_dir = active_experiment_dir / LEVEL_NAME
        if parent_id is not None:
            trained_slot_dir = trained_slot_dir / f"{LEVEL_NAME}_{parent_id}"
        print(f"\n=== 训练 {LEVEL_NAME}  parent={parent_id}  -> {trained_slot_dir} ===")
        run_training(
            TrainRequest(
                config=config,
                level_name=LEVEL_NAME,
                parent_id=parent_id,
                global_head_enable=GLOBAL_HEAD_ENABLE,
                experiment_dir=active_experiment_dir,
            )
        )
        trained_slot_dirs.append(trained_slot_dir)
    LAST_TRAINED_SLOT_DIR = trained_slot_dirs[-1]
    print("\n已训练槽位：")
    for slot_dir in trained_slot_dirs:
        print(f"  {slot_dir}")
    print(f"last_trained_slot_dir = {LAST_TRAINED_SLOT_DIR}")

    # 训练 workflow 已在实验根保存 train_data.npz；这里确认快照完整可用。
    snapshot_path = EXPERIMENT_DIR / TRAIN_ARCHIVE_NAME
    if not snapshot_path.is_file():
        raise RuntimeError(f"缺少训练数据快照：{snapshot_path}")
    with np.load(snapshot_path, allow_pickle=False) as snapshot:
        snapshot_sample_count = int(snapshot["sample_count"])
        intensity_shape = snapshot["intensities"].shape
    print(
        f"train snapshot: {snapshot_path}; "
        f"samples={snapshot_sample_count}; "
        f"shape={intensity_shape}"
    )

## 训练后独立测试集评估

每次评估一个模型槽位，只使用该模型类别范围内的测试文件夹：首层模型评估所有测试文件夹；父类子模型只评估属于该父类的子类文件夹。结果写入该槽位的 `test_result/`，包含 `metrics.json`、`summary.txt`、`confusion_matrix.png` 和 `test_eval_results.csv`；文件夹级指标在 `metrics.json` 的 `folder` 字段中，`summary.txt` 为统一格式的可读摘要。

评估完成后会在同一次流程中补齐层级 `test_result` 下的 `inference_result/`（全局模型为 `<level>/test_result/inference_result/`，多模型层级为 `<level>/level_summary/cascade/test_result/inference_result/`）：已训练完整的层级自动生成，模型槽位未补齐的层级跳过。level_1 在全局测试评估时生成；level_2 要等所有父类模型补齐后，由任一次父类 `test_result` 生成触发真级联。

独立测试主要看两个指标：

- **Spectrum Accuracy**：逐条光谱的 top-1 预测准确率；
- **Folder Accuracy**：每个文件夹按 cell 抽取后，直接取票数最多的输出标签作为文件夹输出；输出标签与文件夹预期标签一致才计为正确。

另外输出 Class Coverage（测试集覆盖的类别数 / 模型类别数）、Macro F1、Macro Recall 作为辅助参考。

评估与样本推理口径一致：每个测试文件夹按 cell 分组随机抽取最多 10 条光谱，同一 cell 至多一条；文件夹内 cell 不足 10 个时逐 cell 取一条。

测试集目录是扁平结构（一层文件夹，命名 `{前缀}{日期}_{来源}`），属名由文件夹前缀反查；不在当前模型类别空间内的文件夹不参与该模型的评估。

In [ ]:
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.independent_test import evaluate_test_set

# 评估刚训练的那个模型槽位；也可手动填写具体槽位目录。
TEST_EVAL_ENABLE = True
TEST_SLOT_DIR = LAST_TRAINED_SLOT_DIR
CPU_ENABLE = False
# None 时使用 dataset/<数据集>/profiles/<profile>/test；填写路径时评估指定的测试集目录。
TEST_DIR_OVERRIDE = None

if TEST_EVAL_ENABLE:
    if TEST_SLOT_DIR is None:
        raise ValueError("请先完成训练，或手动填写 TEST_SLOT_DIR")
    test_dir = (
        Path(TEST_DIR_OVERRIDE)
        if TEST_DIR_OVERRIDE
        else dataset_dir / profile.root_test
    )
    if not test_dir.is_dir():
        print(f"未找到预处理后的测试集：{test_dir}，请先执行 build_test。")
    else:
        test_report = evaluate_test_set(
            TEST_SLOT_DIR,
            "cpu" if CPU_ENABLE else None,
            test_dir,
        )
        print("\n===== 模型独立测试 =====")
        # 结果字典中的槽位路径键为 run_dir。
        print(f"slot_dir          = {test_report['run_dir']}")
        print(
            f"level_name        = {test_report['level_name']}"
            f"  parent_id = {test_report['parent_id']}"
        )
        print(
            f"Spectrum Accuracy = {test_report['spectrum_accuracy'] * 100:.4f}%"
            f"  (support={test_report['support']})"
        )
        print(
            f"Folder Accuracy   = {test_report['folder_accuracy'] * 100:.4f}%"
            f"  ({test_report['folder_correct']}/{test_report['folder_count']})"
        )
        print(
            f"Class Coverage    = {test_report['covered_class_count']}"
            f"/{test_report['class_count']}"
        )
        print(f"Macro F1          = {test_report['macro_f1'] * 100:.4f}%")
        print(f"Macro Recall      = {test_report['macro_recall'] * 100:.4f}%")
        print(f"result_dir        = {test_report['result_dir']}")
        print("========================")
        result_dir = Path(test_report["result_dir"])
        print(f"metrics.json = {result_dir / 'metrics.json'}")
        display(Image(filename=str(result_dir / "confusion_matrix.png")))

## 单模型评估（验证集）

默认评估刚训练的模型槽位，层级与父类索引由实验元数据自动识别。每次评估一个模型槽位，结果写入该槽位的 `val_result/`，包含 `metrics.json`、`confusion_matrix.png` 和 `val_eval_results.csv`；PCA-SVM 基线写入同级 `baseline_val_result/`。评估只使用该模型类别范围内的验证样本。

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.context import resolve_slot_target
from ramanv2.evaluation.pca_svm import BaselineSpec, evaluate_baseline_run
from ramanv2.evaluation.model_eval import evaluate_model_run

# 默认评估刚训练的槽位；也可手动填写其他模型槽位目录。
SINGLE_SLOT_DIR = LAST_TRAINED_SLOT_DIR
SINGLE_EVALUATE_ENABLE = True
CPU_ENABLE = False


def print_validation_metrics(result_name, result_dir):
    payload = json.loads(
        (result_dir / "metrics.json").read_text(encoding="utf-8")
    )
    summary = payload["summary"]
    print(f"\n{result_name} 验证集指标")
    print("=====================================")
    print(f" Val Set Accuracy:  {summary['accuracy'] * 100:.4f}%")
    print(f" Macro F1-score:    {summary['macro_f1'] * 100:.4f}%")
    print(f" Macro Recall:      {summary['macro_recall'] * 100:.4f}%")
    print("=====================================")


if SINGLE_EVALUATE_ENABLE:
    if SINGLE_SLOT_DIR is None:
        raise ValueError("请先完成训练，或手动填写 SINGLE_SLOT_DIR")
    resolved_single_slot_dir = Path(SINGLE_SLOT_DIR)
    single_level, single_parent = resolve_slot_target(resolved_single_slot_dir)
    print(f"slot_dir   = {resolved_single_slot_dir}")
    print(f"level_name = {single_level}  parent_id = {single_parent}")
    device = "cpu" if CPU_ENABLE else None
    single_model_result_dir = evaluate_model_run(
        resolved_single_slot_dir,
        single_level,
        device,
    )
    single_baseline_result_dir = evaluate_baseline_run(
        resolved_single_slot_dir,
        single_level,
        BaselineSpec(),
    )
    print(single_model_result_dir)
    print(single_baseline_result_dir)
    for result_name, result_dir in (
        ("模型", single_model_result_dir),
        ("PCA-SVM 基线", single_baseline_result_dir),
    ):
        figure_path = result_dir / "confusion_matrix.png"
        print_validation_metrics(result_name, result_dir)
        print(f"{result_name} 混淆矩阵：{figure_path}")
        display(Image(filename=str(figure_path)))

## 层级汇总

多父类层级会生成两种模式的验证/测试汇总：

- `parent_known`：已知上层，按真实父类调用本层模型；
- `cascade`：未知上层，从顶层开始逐层推理。

分别写入 `<level>/level_summary/parent_known/` 和 `<level>/level_summary/cascade/`，各自包含 `val_result/` 与 `test_result/`。首层没有上层，只有一套结果，不生成 `level_summary`。


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display
from ramanv2.evaluation.level_summary import summarize_level_results
from ramanv2.modeling.run_manifest import find_model_slot_dirs

# 仅用于非顶层、有父类模型槽位的层级；level_1 全局模型和扁平数据集不需要。
SUMMARY_LEVEL = "level_2"
SUMMARY_ENABLE = True
SUMMARY_CPU_ENABLE = False
# 缺失的父类验证/测试结果会自动评估一次；设为 False 则要求先手动评估。
SUMMARY_AUTO_EVALUATE = True
MODE_TITLES = {
    "parent_known": "已知上层",
    "cascade": "从头级联",
}


def level_summary_available():
    if EXPERIMENT_DIR is None:
        return False, "请先完成训练或填写 EXPERIMENT_DIR"
    meta_path = Path(EXPERIMENT_DIR) / "hierarchy_meta.json"
    if not meta_path.is_file():
        return False, f"缺少实验元数据：{meta_path}"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    if SUMMARY_LEVEL not in (meta.get("head_names") or []):
        return False, f"实验没有 {SUMMARY_LEVEL}，扁平数据集不需要层级汇总"
    global_entry = (meta.get("level_models") or {}).get(SUMMARY_LEVEL)
    if isinstance(global_entry, dict) and global_entry.get("model_path"):
        return False, f"{SUMMARY_LEVEL} 是全局模型层级，不需要层级汇总"
    slots = find_model_slot_dirs(Path(EXPERIMENT_DIR) / SUMMARY_LEVEL)
    if not slots:
        return False, f"{SUMMARY_LEVEL} 没有父类模型槽位"
    return True, ""


SUMMARY_AVAILABLE, SUMMARY_REASON = level_summary_available()
if SUMMARY_ENABLE and not SUMMARY_AVAILABLE:
    print(f"跳过层级汇总：{SUMMARY_REASON}")
elif SUMMARY_ENABLE:
    summary_report = summarize_level_results(
        EXPERIMENT_DIR,
        SUMMARY_LEVEL,
        "cpu" if SUMMARY_CPU_ENABLE else None,
        auto_evaluate_enable=SUMMARY_AUTO_EVALUATE,
    )
    for mode in ("parent_known", "cascade"):
        for output_name, title in (("val_result", "验证"), ("test_result", "测试")):
            result_dir = Path(summary_report["level_summary_dir"]) / mode / output_name
            metrics_path = result_dir / "metrics.json"
            if not metrics_path.is_file():
                print(f"{MODE_TITLES[mode]} {title}结果尚未生成：{result_dir}")
                continue
            metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
            summary = metrics["summary"]
            print()
            print(f"===== {MODE_TITLES[mode]} · {title} =====")
            print(f"Accuracy = {summary['accuracy'] * 100:.4f}%  (support={summary['support']})")
            print(f"Macro F1 = {summary['macro_f1'] * 100:.4f}%")
            print(f"Macro Recall = {summary['macro_recall'] * 100:.4f}%")
            print(f"result_dir = {result_dir}")
            display(Image(filename=str(result_dir / "confusion_matrix.png")))

## 单模型 Analysis

AutoDL: 如果这里提示缺少 umap，请回到环境单元把 `INSTALL_UMAP_DEPS` 设为 `True` 并重新运行该单元。


In [ ]:
from IPython.display import Image, display
from ramanv2.analysis.interpret import interpret_slot
from ramanv2.evaluation.context import resolve_slot_target

SINGLE_ANALYZE_ENABLE = True
CPU_ENABLE = False
# 归因只抽取最多 5 个 batch；IG 步数仍沿用实验配置。
ANALYSIS_ATTRIBUTION_BATCH_COUNT = 5

# 默认分析刚训练的槽位；“单模型评估”格若设置了 SINGLE_SLOT_DIR 则优先使用。
resolved_single_slot_dir = globals().get("SINGLE_SLOT_DIR") or globals().get("LAST_TRAINED_SLOT_DIR")
if SINGLE_ANALYZE_ENABLE:
    if resolved_single_slot_dir is None:
        raise ValueError("请先训练一个唯一模型，或填写 SINGLE_SLOT_DIR")
    single_level, single_parent = resolve_slot_target(resolved_single_slot_dir)
    print(f"slot_dir   = {resolved_single_slot_dir}")
    print(f"level_name = {single_level}  parent_id = {single_parent}")
    single_analysis_result_dir = interpret_slot(
        resolved_single_slot_dir,
        single_level,
        "cpu" if CPU_ENABLE else None,
        attribution_batch_count=ANALYSIS_ATTRIBUTION_BATCH_COUNT,
    )
    print(single_analysis_result_dir)
    figure_dir = single_analysis_result_dir / "figures"
    for figure_name in ("channel_importance_IG.png", "layer_importance.png"):
        figure_path = figure_dir / figure_name
        print(figure_path)
        display(Image(filename=str(figure_path)))
    analysis_log_path = single_analysis_result_dir / "logs" / "analysis_log.txt"
    if analysis_log_path.is_file():
        print("\n===== Analysis 摘要 =====")
        print(analysis_log_path.read_text(encoding="utf-8"))
    else:
        print("未生成 Analysis 摘要日志。")

将完整实验目录压缩为 ZIP，包含模型、验证/测试结果、级联推理结果、Analysis 和 `train_data.npz` 快照。压缩包落在项目根目录，路径与 `sha256` 会打印出来。可在文件树中下载，或通过 `scp` 拉回本地，用 `sha256` 校验传输完整性。

In [ ]:
import hashlib
import shutil
from pathlib import Path

# False 时只打印实验目录路径，不生成压缩包。
PACKAGE_RESULT_ENABLE = True


def compute_file_sha256(path: Path) -> str:
    """分块计算文件 sha256，避免大文件整体读入内存。"""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def publish_experiment_archive(experiment_dir: Path) -> Path | None:
    """把实验目录压缩到项目根目录，并打印取回所需信息。"""
    if not experiment_dir.is_dir():
        raise FileNotFoundError(f"实验目录不存在：{experiment_dir}")
    print(f"experiment_dir = {experiment_dir}")
    if not PACKAGE_RESULT_ENABLE:
        return None
    archive_path = PROJECT_ROOT / f"{experiment_dir.name}.zip"
    if archive_path.exists():
        archive_path.unlink()
    # root_dir + base_dir 保留实验目录这一层，解压后直接得到完整实验目录。
    shutil.make_archive(
        str(archive_path.with_suffix("")),
        "zip",
        root_dir=experiment_dir.parent,
        base_dir=experiment_dir.name,
    )
    print(f"experiment_archive = {archive_path}")
    print(f"size = {archive_path.stat().st_size / 2**20:.1f} MiB")
    print(f"sha256 = {compute_file_sha256(archive_path)}")
    print("取回方式：Jupyter 文件树下载，或 scp 拉取该路径。")
    return archive_path


if EXPERIMENT_DIR is None:
    raise ValueError("请先完成训练或填写 EXPERIMENT_DIR")
result_archive = publish_experiment_archive(Path(EXPERIMENT_DIR).resolve())
